# Kapitel 2 – Övningsuppgifter: Ett ML-projekt från början till slut

Mina svar på övningsuppgifterna till kapitel 2 i *"Lär dig AI från grunden - Tillämpad maskininlärning med Python"* (Prgomet, Johnson, Solberg, Rundberg Streuli), baserade på boken och uppgifterna som ligger i bokens GitHub-repo (https://github.com/AntonioPrgomet/ai_tillaempad_ml).

## Fråga 1 – Checklistans sju steg, rak progression eller iterativt?

Boken går igenom en checklista för maskininlärningsprojekt i Avsnitt 2.1, och den består av sju steg:

1. Definiera problemet och skapa en helhetsbild – förstå vad målet med projektet egentligen är, om det man bygger kommer vara användbart, om det redan finns lösningar eller "workarounds", hur man mäter om projektet lyckats, och vilka personer som är bra att ha med på tåget.
2. Få tillgång till data – skaffa den data som behövs och se till att den hanteras lagenligt, t.ex. genom att anonymisera skyddade uppgifter. Redan i det här steget bör man lägga undan testdata (och ofta tränings- och valideringsdata också) som inte rörs förrän helt i slutet.
3. Utforska datan, gör en EDA (*exploratory data analysis*) – på en kopia av träningsdatan. Man räknar ut medelvärden, medianer och typvärden, visualiserar för att hitta samband och mönster, och letar efter fel och saknade värden.
4. Bearbeta datan – *data cleaning* (hantera outliers och saknade värden), ta bort variabler som inte tillför något, och *feature engineering* (skapa nya variabler eller transformera befintliga). Det är smart att skriva funktioner för det här, dels för att kunna återanvända dem på ny data, dels för att kunna behandla själva data-transformerna som en hyperparameter i en `GridSearch`.
5. ML-modellering – man provar först flera modeller med standardvärden på hyperparametrarna, utvärderar dem på valideringsdatan, och justerar sedan hyperparametrar och variabler iterativt för de mest lovande kandidaterna. Till sist utvärderas den bästa modellens generaliseringsförmåga på testdatan, som ett sista steg.
6. Presentera lösningen för intressenter – man anpassar abstraktionsnivå och budskap efter vem som lyssnar, tekniska detaljer för kollegor men konkreta resultat och ekonomiska konsekvenser för en ledningsgrupp, gärna med hjälp av visualiseringar.
7. Produktionsättning av modellen och övervakning av implementeringen – se Fråga 2.

Det är lätt att få intrycket att de här stegen görs i rak följd, ett efter ett, men boken är tydlig med att så inte är fallet i verkligheten. Man jobbar mer **iterativt**, hoppar fram och tillbaka mellan stegen. Ett exempel: i steg 2, när man ska skaffa data, kan man upptäcka att den data som krävs för det ursprungliga målet helt enkelt inte finns, och då tvingas man tillbaka till steg 1 för att fundera på om målsättningen behöver ändras. Checklistan är ett stöd i arbetet, inte något man måste följa slaviskt från topp till botten.

## Fråga 2 – Vad menas med att en modell produktionsätts?

Att en modell produktionsätts betyder att den färdigtränade modellen faktiskt börjar användas skarpt i den verksamhet eller det system den var tänkt för, istället för att bli kvar som ett experiment i en notebook. Checklistans sjunde och sista steg, "Produktionsättning av modellen och övervakning av implementeringen" (Avsnitt 2.1.7), handlar bland annat om att:

- skapa enhetstester för att säkerställa att funktionaliteten fungerar som den ska,
- bevaka modellens prestanda över tid, eftersom modeller tenderar att bli sämre allt eftersom datan förändras (till exempel på grund av förändringar inom innovation, ekonomi och människors beteende),
- träna om modellen regelbundet på ny data, kanske månadsvis eller veckovis, för att hålla den aktuell.

En praktisk del av det här är att kunna **spara** den tränade modellen, till exempel med `joblib` eller `pickle` (se Fråga 8), så att den går att ladda in och återanvända för prediktioner utan att behöva tränas om varje gång. Utan det skulle det knappast gå att integrera modellen i ett skarpt system.

## Fråga 3 – Scikit-learn, designprinciper, estimators/predictors/transformers

Scikit-learn är ett open source-bibliotek för maskininlärning i Python (Avsnitt 2.4), med algoritmer för bland annat regression, klassificering, klustring, modellval och databearbetning. Det lanserades 2010 och är idag det mest använda ML-biblioteket, både i näringslivet och akademin – enligt Figur 2.2 i boken uppgav 82,8% av tillfrågade personer inom data science-branschen att de använde scikit-learn år 2020, klart mest av alla bibliotek som togs upp. Namnet kommer från att projektet från början var tänkt som ett "scientific toolkit for machine learning".

Scikit-learn är byggt för att vara enkelt, effektivt och tillgängligt även för den som inte är expert, och det vilar på fyra designprinciper (Avsnitt 2.4.1). Konsistens är den första: alla objekt har samma sorts gränssnitt och ett begränsat antal metoder, så oavsett om man tränar en `LinearRegression` eller en `DecisionTreeRegressor` går det till exakt likadant – man instantierar modellen, tränar den med `.fit()`, och gör prediktioner med `.predict()`. Även dokumentationen är strukturerad likadant för alla modeller. Den andra är inspektion: hyperparametrar (kallas "parametrar" i scikit-learns dokumentation) nås alltid med `.get_params()`, och lärda parametrar (kallas "attribut") går alltid att nå som publika attribut med understräck på slutet, till exempel `.coef_` och `.intercept_` för en `LinearRegression`. Den tredje är begränsning av klasser: bara själva modellalgoritmerna representeras som byggda klasser, medan data representeras som NumPy arrays eller SciPy sparse matrices och hyperparametrar som vanliga strängar eller siffror när det går. Den fjärde är rimliga standardvärden för hyperparametrarna – man behöver inte förstå alla teoretiska detaljer för att komma igång, eftersom scikit-learn redan har satt ett vettigt default-värde för varje hyperparameter.

Sen har vi begreppen estimators, predictors och transformers (Avsnitt 2.4.2). En estimator är det centrala objektet i scikit-learn, i praktiken vilket objekt som helst som lär sig (estimerar) något från data via `.fit()`, till exempel `LinearRegression()`. En predictor är en delmängd av estimators, nämligen de som kan göra prediktioner via `.predict()` och som dessutom alltid har en `.score()`-metod som talar om hur bra prediktionerna är (i scikit-learn gäller alltid att högre score är bättre). `LinearRegression` är alltså både en estimator och en predictor. En transformer är också en delmängd av estimators – de som kan transformera data via `.transform()`, som `StandardScaler()`. Transformers har dessutom bekvämlighetsmetoden `.fit_transform()`, som tränar och transformerar i ett enda steg.

En kedja av flera databearbetningssteg som avslutas med en transformer eller predictor kallas en **pipeline** (Avsnitt 2.4.3). Fördelen är att man bara behöver anropa `.fit()`/`.predict()` en gång för hela kedjan, och kan grid-söka över samtliga steg samtidigt.

## Fråga 4 – Vad är TensorFlow och Keras?

Enligt Avsnitt 2.4.4 är TensorFlow och Keras de bibliotek som, tillsammans med scikit-learn, ligger i topp enligt undersökningen i Figur 2.2 – de delar en andraplats med 50,5% användning vardera bland yrkesverksamma inom data science.

TensorFlow är motorn, en plattform för framförallt djupinlärning som sköter de tunga beräkningarna. Keras beskrivs i boken som "the high-level API for TensorFlow" – ett lättillgängligt och produktivt gränssnitt för att lösa ML-problem med fokus på modern djupinlärning, som täcker hela arbetsflödet från databearbetning till hyperparameter-tuning och produktionsättning. Enligt TensorFlow-dokumentationen, som boken citerar, bör "every TensorFlow user" använda Keras-API:erna som standard, oavsett om man är ingenjör, forskare eller ML-praktiker.

Boken sammanfattar det hela ganska fint med en bilmetafor: **Keras är ratten och TensorFlow är motorn**. Scikit-learn har visserligen också en del funktionalitet för djupinlärning, men den är inte i närheten lika sofistikerad som TensorFlow/Keras på det området. TensorFlow och Keras dyker upp på allvar från och med bokens tredje del (Djupinlärning), med start i Kapitel 7.

## Fråga 5 – Kalle och Stinas dialog om att justera modellen efter testdatan

Kalle: *"om jag tränat en modell och den inte presterar bra nog på testdatan så justerar jag den tills den gör det."* Stina: *"det är ett stort fel att göra så, det enda du då åstadkommer är att du överanpassar testdatan. Hela syftet med testdatan försvinner då."*

Stina har rätt, och boken varnar faktiskt uttryckligen för precis det här i informationsrutan "Överanpassa inte modellen till testdatan" (Avsnitt 2.1.5). Testdatan ska användas som ett sista steg, för att ge en så rättvis skattning som möjligt av modellens **generaliseringsförmåga** på helt ny, osedd data, efter att modellval och hyperparameter-justering redan är klara på valideringsdatan. Om man som Kalle går tillbaka och justerar modellen, byter hyperparametrar eller väljer en annan modell, efter att man redan sett resultatet på testdatan, så överanpassar man i praktiken modellen till just den datan. Testdatan slutar då fungera som en oberoende skattning eftersom den indirekt har använts för modellval, och det RMSE (eller vilket mått man nu använder) man till slut rapporterar blir missvisande optimistiskt. Hela poängen med att lägga undan testdata redan i checklistans steg 2 (Avsnitt 2.1.2) går då förlorad.

Boken skriver rakt ut att "är vi inte nöjda behöver vi börja om från början", och gör en tydlig skillnad mellan att faktiskt börja om (ny modellering på tränings- och valideringsdata, med en ny och opåverkad testomgång i slutet) och att bara pilla på hyperparametrarna tills modellen råkar prestera bra nog på testdatan. Så Stinas kritik stämmer väl med bokens resonemang. Är Kalle inte nöjd med resultatet är rätt väg att gå tillbaka till steg 5 i checklistan (ML-modellering) och jobba vidare på tränings- och valideringsdatan, för att sedan, när man känner sig klar, göra en ny slutgiltig utvärdering på testdata – eller i strängaste mening skaffa helt ny testdata om den gamla känns förbrukad.

## Fråga 6 – Varför misslyckas många AI/ML-projekt, och hur bör vi förhålla oss till det?

Boken tar upp det här rätt rakt på sak i informationsrutan "De flesta AI/ML-projekten misslyckas" i inledningen till Avsnitt 2.2. Det cirkulerar olika uppskattningar om att omkring 85% av alla ML-projekt misslyckas, i den meningen att de varken når de mål som sattes upp från början eller ens tar sig förbi någon form av prototypstadium. Några vanliga orsaker som boken nämner är brist på behövlig data, dåliga modeller, eller att rätt kompetens och resurser saknas. Det hänger ganska tydligt ihop med de tidiga stegen i checklistan (Avsnitt 2.1.1–2.1.2) – har man inte skapat sig en tillräcklig helhetsbild, eller inte har tillgång till rätt eller tillräckligt mycket data från början, är risken stor att projektet fastnar. Det kopplar också till Avsnitt 2.3 om utmaningar inom ML, där för lite data, icke-representativ data, dålig datakvalitet och irrelevanta features ("shit in - shit out") lyfts fram som vanliga fällor.

Så hur ska man då förhålla sig till det här? Boken landar i att man bör ha rimliga förväntningar – det är inte rimligt att tro att varje projekt man jobbar med faktiskt kommer nå ända fram till att användas skarpt. Man bör inte heller tänka för binärt i termer av att ett projekt antingen "lyckas" eller "misslyckas", utan snärare fundera kreativt på vad som ändå gick att åstadkomma eller lära sig. Kanske blev slutmodellen inte tillräckligt bra, men EDA:n som gjordes är fortfarande värdefull för verksamheten, eller så har man i alla fall lärt sig vad som krävs, till exempel mer eller bättre data, för att lyckas nästa gång. Poängen är egentligen att se varje projekt som en möjlighet att skapa värde och lärdomar, även om det ursprungliga målet inte nås fullt ut, snärare än att stämpla det som ett rent lyckat eller misslyckat facit.

## Fråga 8 – Spara och ladda en modell med joblib

```python
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression
from joblib import dump, load

X, y = make_regression(n_samples=20000, n_features=3, noise=0.1)

model = LinearRegression().fit(X, y)
dump(model, "linear_model.joblib")
model_loaded = load("linear_model.joblib")

print(model_loaded.predict(X[:5]))
```

Går man igenom koden rad för rad: `make_regression(...)` skapar ett syntetiskt regressionsdataset med 20 000 observationer, 3 features och lite brus (`noise=0.1`), och returnerar `X` (features) och `y` (beroende variabel). `LinearRegression().fit(X, y)` instantierar en linjär regressionsmodell och tränar den direkt, i samma rad, på hela datasetet. `dump(model, "linear_model.joblib")` sparar den tränade modellen till filen `linear_model.joblib`, i samma mapp som skriptet körs från (jämför Avsnitt 2.1.7 om att spara modeller med joblib/pickle). `load("linear_model.joblib")` laddar sedan in den sparade modellen igen som `model_loaded` – i praktiken skulle det här kunna ske i ett helt annat skript eller program, vid ett senare tillfälle, utan att modellen behöver tränas om. Sist använder `model_loaded.predict(X[:5])` den inladdade modellen för att göra prediktioner på de fem första observationerna.

Varför är det här viktigt? Att träna om en modell från grunden varje gång den ska användas är ofta både tidskrävande och onödigt kostsamt, särskilt för mer komplexa modeller eller stora datamängder. Genom att **spara** den färdigtränade modellen (produktionsätta den, se Fråga 2) kan man istället ladda in den direkt och återanvända den för nya prediktioner, vilket är en förutsättning för att kunna integrera en ML-modell i ett skarpt system eller en produkt utan orimlig väntetid eller beräkningskostnad.

## Fråga 9 – Tränings-, validerings- och testset samt modellval steg för steg

a) Läsa in `data_01.csv` med `pd.read_csv()`, vilket ger en `DataFrame`.

b) Dela upp datasetet i `X` (de oberoende variablerna) och `y` (den beroende variabeln).

c) Dela upp datan ytterligare, i tränings-, validerings- och testset, genom att använda `train_test_split()` två gånger efter varandra (jämför resonemanget om `test_size` i Kapitel 1 samt kodexemplet i Avsnitt 2.2): först `test_size=0.20` för att avskilja 20% testdata, och sedan `test_size=0.15` på det som blev kvar för att avskilja valideringsdata (15% av resten).

d) Träna två valfria regressionsmodeller, till exempel `LinearRegression` och `DecisionTreeRegressor`, på träningsdatan.

e) Utvärdera båda modellerna på valideringsdatan, till exempel med RMSE, för att se vilken som presterar bäst (jämför principen "högre är bättre" med `neg_root_mean_squared_error`, se Kapitel 1, Fråga 6).

f) Träna om den bäst presterande modellen på tränings- och valideringsdatan tillsammans – samma princip som används i bokens eget kodexempel (Avsnitt 2.2).

g) Utvärdera den omtränade modellen på testdatan, som ett sista och orört mått på generaliseringsförmåga (se Fråga 5 för resonemanget kring varför man inte ska fortsätta justera modellen efter det här steget).

h) Till sist tränas modellen om en sista gång på hela datasetet, inför en eventuell produktionsättning (jämför checklistans steg 7, Fråga 2).

*Datasetet `data_01.csv` fanns inte tillgängligt lokalt när den här notebooken skapades och skulle behöva laddas ner från bokens GitHub-repo (mapparna `material/` eller `övningsuppgifter/dataset/` på https://github.com/AntonioPrgomet/ai_tillaempad_ml) för att koden nedan faktiskt ska gå att köra. Koden antar att den beroende variabeln heter `"y"` i datasetet – byt kolumnnamn vid behov.*

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import root_mean_squared_error

# a) Läs in datasetet
df = pd.read_csv("data_01.csv")

# b) Dela upp i X och y (antag att den beroende variabeln heter "y")
X = df.drop(columns=["y"])
y = df["y"]

# c) Dela upp i tränings-, validerings- och testset (20% test, 15% av resten som validering)
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.15, random_state=42
)

# d) Träna två regressionsmodeller på träningsdatan
lin_model = LinearRegression()
tree_model = DecisionTreeRegressor(random_state=42)
lin_model.fit(X_train, y_train)
tree_model.fit(X_train, y_train)

# e) Utvärdera modellerna på valideringsdatan
lin_pred_val = lin_model.predict(X_val)
tree_pred_val = tree_model.predict(X_val)
print("RMSE Linear Regression (val):", root_mean_squared_error(y_val, lin_pred_val))
print("RMSE Decision Tree (val):    ", root_mean_squared_error(y_val, tree_pred_val))

# f) Träna om den bäst presterande modellen (anta här att det är Linear Regression,
#    byt ut vid behov) på tränings- och valideringsdatan tillsammans
best_model = LinearRegression()
best_model.fit(X_train_full, y_train_full)

# g) Utvärdera den ombtränade modellen på testdatan
test_pred = best_model.predict(X_test)
print("RMSE på testdata:", root_mean_squared_error(y_test, test_pred))

# h) Träna om modellen på hela datasetet, inför produktionssättning
final_model = LinearRegression()
final_model.fit(X, y)

## Fråga 10 – Salary_dataset.csv med k-delad korsvalidering

a) Läsa in datasetet med `pd.read_csv()` och dela upp i `X` och `y`. Eftersom vi vill predicera lönen utifrån antal års erfarenhet är `Salary` den beroende variabeln (`y`), och `YearsExperience` den oberoende variabeln (`X`).

b) Dela upp datasetet i tränings- och testset med `train_test_split()`. Inget valideringsset behövs här eftersom vi istället använder k-delad korsvalidering på träningsdatan för att jämföra modeller (jämför Kapitel 1, Fråga 3b).

c) Träna två regressionsmodeller med k-delad korsvalidering via `cross_validate()`, med `scoring="neg_root_mean_squared_error"` (eftersom "högre är bättre" gäller för scikit-learns scoring, se Kapitel 1, Fråga 6 och 9). Antal iterationer (`cv`) sätts här till 5.

d) Den modell som presterar bäst i korsvalideringen (lägst genomsnittligt RMSE) tränas sedan på hela träningsdatan och utvärderas slutligen på testsetet.

*Datasetet `salary_dataset.csv` fanns inte tillgängligt lokalt när den här notebooken skapades och skulle behöva laddas ner från bokens GitHub-repo (`övningsuppgifter/dataset/` på https://github.com/AntonioPrgomet/ai_tillaempad_ml) för att koden nedan faktiskt ska gå att köra.*

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import root_mean_squared_error

# a) Läs in datasetet och dela upp i X och y
df = pd.read_csv("salary_dataset.csv")
X = df[["YearsExperience"]]
y = df["Salary"]  # beroende variabel

# b) Dela upp i tränings- och testset (inget valideringsset - vi använder k-delad korsvalidering istället)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# c) K-delad korsvalidering med två modeller
lin_model = LinearRegression()
tree_model = DecisionTreeRegressor(random_state=42)

lin_cv = cross_validate(
    lin_model, X_train, y_train, cv=5, scoring="neg_root_mean_squared_error"
)
tree_cv = cross_validate(
    tree_model, X_train, y_train, cv=5, scoring="neg_root_mean_squared_error"
)

print("Linear Regression, medel-RMSE (CV):", -np.mean(lin_cv["test_score"]))
print("Decision Tree, medel-RMSE (CV):    ", -np.mean(tree_cv["test_score"]))

# d) Utvärdera den bäst presterande modellen (anta här Linear Regression, byt ut vid behov) på testsetet
best_model = LinearRegression()
best_model.fit(X_train, y_train)
test_pred = best_model.predict(X_test)
print("RMSE på testdata:", root_mean_squared_error(y_test, test_pred))

## Fråga 11 – Kategorisk data: mpg-datasetet

a) Läsa in datasetet `mpg` med seaborns `load_dataset()`-funktion. Det innehåller 398 observationer av bilar, där `mpg` (miles per gallon) är den beroende variabeln.

b) Droppa rader med saknade värden med `dropna()` (jämför data cleaning, Avsnitt 2.1.4).

c) Droppa kolumnen `name`, som har 305 unika värden och därför inte passar för encoding – den skulle ge orimligt många nya kolumner och blir inte generaliserbar (en bilmodell som saknas i träningsdatan går ju inte att hantera).

d) `origin` är en nominal kategorisk variabel (europe/japan/usa, utan inbördes rangordning) och passar därför bra för encoding. Eftersom vi ska använda linjär regression används dummy-variable-encoding med `pd.get_dummies(..., drop_first=True)`, vilket ger 2 nya kolumner istället för 3 (jämför Kapitel 1, Fråga 3f).

e) Dela upp datasetet i `X` och `y`, med `mpg` som beroende variabel.

f) Dela upp i tränings- och testset med `train_test_split()`.

g) Träna en `LinearRegression` på träningsdatan och utvärdera den (RMSE) på testdatan.

Koden nedan kräver ingen lokal fil, seaborn laddar ner `mpg`-datasetet automatiskt, och den har körts här för att visa verklig utdata.

In [1]:
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

# a) Läs in datasetet
df = sns.load_dataset("mpg")
print(df.head())

# b) Droppa rader med saknade värden
df = df.dropna()

# c) Droppa kolumnen name
df = df.drop(columns=["name"])

# d) Dummy-variable-encoding på origin (drop_first=True ger 2 kolumner istället för 3)
df = pd.get_dummies(df, columns=["origin"], drop_first=True)

# e) Dela upp i X och y
X = df.drop(columns=["mpg"])
y = df["mpg"]

# f) Dela upp i tränings- och testset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# g) Träna en linjär regressionsmodell och utvärdera på testdatan
model = LinearRegression()
model.fit(X_train, y_train)
preds = model.predict(X_test)
rmse = root_mean_squared_error(y_test, preds)
print("RMSE:", rmse)

    mpg  cylinders  displacement  ...  model_year  origin                       name
0  18.0          8         307.0  ...          70     usa  chevrolet chevelle malibu
1  15.0          8         350.0  ...          70     usa          buick skylark 320
2  18.0          8         318.0  ...          70     usa         plymouth satellite
3  16.0          8         304.0  ...          70     usa              amc rebel sst
4  17.0          8         302.0  ...          70     usa                ford torino

[5 rows x 9 columns]
RMSE: 3.256114096847396


## Fråga 12 – Förbättra RMSE Random Forest Regression (52277.97) i huspris-exemplet

I bokens kodexempel (Avsnitt 2.2) tränades och grid-söktes en `RandomForestRegressor` över rutnätet

```python
hyperparam_grid = {'max_depth': [5, 10, 15, 50], 'n_estimators': [1, 5, 10]}
```

vilket gav RMSE Random Forest Regression: 52277.96578719621 på valideringsdatan, med de optimala hyperparametrarna `{'max_depth': 50, 'n_estimators': 10}`. Här är några bokgrundade sätt man skulle kunna försöka förbättra det resultatet på.

Det första som sticker ut är att grid searchen borde utökas. Både `max_depth=50` och `n_estimators=10` hamnade i kanten av det rutnät som testades, det vill säga det högsta värdet som provades för respektive hyperparameter. Det är ett ganska tydligt tecken på att den bästa modellen kan finnas utanför det rutnät man sökte igenom – man skulle till exempel kunna testa fler och högre värden på `n_estimators` (50, 100, 200) och fler värden runt `max_depth=50`. Det är också värt att söka över fler hyperparametrar än de två boken använde, som `min_samples_leaf`, `min_samples_split` och `max_features`.

Sen finns variabelselektion. I bokens korrelationsanalys (Avsnitt 2.2) hade variabler som `population`, `longitude` och `total_bedrooms` mycket svag korrelation med `median_house_value`, nära 0. Boken valde ändå medvetet att använda alla variabler "för att begränsa kodexemplets omfattning", men man skulle kunna testa att plocka bort de svagast korrelerade variablerna, eller göra en mer systematisk variabelselektion, och se om modellen blir enklare eller bättre av det (jämför principle of parsimony, Kapitel 1, Fråga 3h).

En tredje väg är feature engineering. Boken skapar inga nya variabler i huspris-exemplet, det enda som görs är dummy-variable-encoding av `ocean_proximity`. Att istället skapa nya, mer informativa kvot-variabler utifrån de befintliga, till exempel `rooms_per_household`, `bedrooms_per_room` och `population_per_household`, är ett klassiskt grepp för att förbättra prediktionsförmågan på just det här datasetet, eftersom sådana kvoter ofta fångar mönster som de enskilda variablerna missar var för sig (jämför feature engineering, Avsnitt 2.1.4).

Man skulle förstås också kunna prova fler eller andra modeller överlag – i bokens exempel jämförs bara `LinearRegression` med en `RandomForestRegressor`, så fler modellalternativ hade kunnat ge en bättre utgångspunkt innan man ens börjar finjustera hyperparametrar.

Nedan är ett kodutkast som visar hur man skulle kunna kombinera en bredare grid search med enkel feature engineering. Koden kräver `housing.csv` (California Housing dataset från bokens hemsida/GitHub-repo eller https://www.kaggle.com/datasets/camnugent/california-housing-prices), som inte fanns tillgängligt lokalt när den här notebooken skapades, och körs därför inte här – inget nytt RMSE-värde påstås, bara tillvägagångssättet demonstreras.

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import root_mean_squared_error

# Antag att X_train, y_train, X_val, y_val redan finns (se bokens Avsnitt 2.2)
housing = pd.read_csv("housing.csv")

# Feature engineering: skapa nya kvot-variabler (jämför Avsnitt 2.1.4)
housing["rooms_per_household"] = housing["total_rooms"] / housing["households"]
housing["bedrooms_per_room"] = housing["total_bedrooms"] / housing["total_rooms"]
housing["population_per_household"] = housing["population"] / housing["households"]

# Variabelselektion: undersök korrelationen med median_house_value och överväg
# att ta bort variabler med mycket svag korrelation (t.ex. population, longitude)
corr_matrix = housing.corr(numeric_only=True)
print(corr_matrix["median_house_value"].sort_values(ascending=False))

# Bredare hyperparameter-grid än i bokens exempel, eftersom de optimala värdena
# i boken (max_depth=50, n_estimators=10) låg i kanten av det ursprungliga rutnätet
# {'max_depth': [5, 10, 15, 50], 'n_estimators': [1, 5, 10]}
hyperparam_grid = {
    "max_depth": [15, 30, 50, 75, 100, None],
    "n_estimators": [10, 50, 100, 200],
    "min_samples_leaf": [1, 2, 4],
    "max_features": [1.0, "sqrt", "log2"],
}

rf = RandomForestRegressor(random_state=42)
grid_search = GridSearchCV(
    rf, hyperparam_grid, scoring="neg_root_mean_squared_error", cv=5, n_jobs=-1
)
grid_search.fit(X_train, y_train)

print("Bästa hyperparametrar:", grid_search.best_params_)

rf_pred_val = grid_search.predict(X_val)
print("RMSE Random Forest (val):", root_mean_squared_error(y_val, rf_pred_val))